# เพิ่มตัวอย่างคลาสส่วนน้อยด้วย SMOTENC

รันหลัง feature_engineering.ipynb เลือก train หรือ train_full เพื่อสร้างชุดฝึกที่เพิ่มตัวอย่างของคลาสเป้าหมาย


## 1. เลือก split และจำนวนตัวอย่าง

กำหนด SMOTE_SPLIT, TARGET_CLASS, TARGET_SAMPLES และ RANDOM_STATE ก่อนรันขั้นต่อไป


In [ ]:
SMOTE_SPLIT = "train"
TARGET_CLASS = 4
TARGET_SAMPLES = 40000
RANDOM_STATE = 42


## 2. เตรียม path และ schema ฟีเจอร์

หาโฟลเดอร์ CENTRAL และกำหนดชื่อคลาส คอลัมน์ตัวเลข และหมวดหมู่สำหรับ SMOTENC


In [ ]:
from pathlib import Path

PIPELINE_DIR = next(
    candidate
    for ancestor in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (ancestor, ancestor / "data-pipeline")
    if (candidate / "CENTRAL" / "feature_contract.json").is_file()
)

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import json
import joblib

NUMERIC = ["duration", "bytes_forward", "bytes_backward"]


CATEGORIES = {
    "protocol": ["tcp", "udp", "icmp", "other"],
    "service_group": ["dns", "ftp", "http", "https_ssl", "smtp", "ssh", "other"],
    "conn_state_group": ["established", "no-reply", "reset", "other"],
}


CORE = NUMERIC + list(CATEGORIES)


PREFIXES = ["proto", "service", "state"]


NAMES = [
    "log_duration",
    "log_bytes_forward",
    "log_bytes_backward",
    "signed_log_bytes_ratio",
] + [
    f"{prefix}_{value}"
    for prefix, values in zip(PREFIXES, CATEGORIES.values())
    for value in values
]


CLASS_NAMES = [
    "Benign / Normal",
    "Denial of Service (DoS)",
    "Reconnaissance & Probing",
    "Exploits & PrivEsc",
    "Credential Abuse (R2L)",
    "Malware, C2 & Botnet",
]


def require(condition, message):
    if not condition:
        raise ValueError(message)


## 3. เตรียมการแปลงฟีเจอร์สำหรับโมเดล

กำหนดวิธีใช้ preprocessor ที่บันทึกไว้ เพื่อให้ผลลัพธ์เป็นฟีเจอร์ 19 คอลัมน์ในลำดับเดิม


In [ ]:
def derived(f):
    x = np.log1p(f[NUMERIC].to_numpy(float))
    return np.column_stack([x, x[:, 1] - x[:, 2]])


def transform_features(f, preprocessor):
    # Validation is shared with resampling; no silent clipping or imputation.
    for c in NUMERIC:
        require(np.isfinite(f[c]).all() and f[c].ge(0).all(), f"Invalid {c}")
    for c, values in CATEGORIES.items():
        require(f[c].isin(values).all(), f"Invalid {c}")
    x = np.column_stack(
        [
            preprocessor["scaler"].transform(derived(f)),
            preprocessor["encoder"].transform(f[list(CATEGORIES)]),
        ]
    )
    require(np.isfinite(x).all(), "Encoded features are not finite")
    result = pd.DataFrame(x, columns=NAMES)
    result.insert(0, "record_id", f.record_id.to_numpy())
    return result


QUALITY_FLAGS = ['invalid_required_value', 'bytes_packet_conflict', 'state_counter_conflict']


## 4. เตรียมตัวอ่านและตรวจตาราง

กำหนดกฎอ่าน CSV ตรวจ record_id และตรวจคุณภาพก่อนและหลังเพิ่มตัวอย่าง


In [ ]:
def normalize_boolean_metadata(metadata):
    columns=[c for c in metadata if c.startswith(('has_','invalid_')) or c in QUALITY_FLAGS+['quality_ok','service_identified','state_reclassified','dedup_flag','is_kddtest21_view']]
    for col in columns:
        values=metadata[col].astype('string').str.lower()
        if not values.isin(['true','false','1','0']).all():raise ValueError(f'Invalid boolean metadata: {col}')
        metadata[col]=values.map({'true':True,'false':False,'1':True,'0':False}).astype(bool)
    return metadata


def text(values):
    return values.astype('string').str.strip().str.lower().replace({'':'', '-':'-'})


def quality_flags(features, metadata):
    f=features
    invalid=pd.Series(False,index=f.index)
    for col in ['duration','bytes_forward','bytes_backward']:
        values=pd.to_numeric(f[col],errors='coerce')
        invalid |= values.isna() | ~np.isfinite(values) | values.lt(0)
    invalid |= f.protocol.isna() | f.protocol.astype('string').str.strip().isin(['','-'])
    packet_conflict=pd.Series(False,index=f.index)
    for direction in ['forward','backward']:
        col='packets_'+direction
        if col in f:
            values=pd.to_numeric(f[col],errors='coerce')
            packet_conflict |= values.eq(0) & f['bytes_'+direction].gt(0)
            invalid |= f[col].notna() & (~np.isfinite(values) | values.lt(0))
    raw_state=text(metadata.get('source_connection_state',pd.Series(pd.NA,index=f.index,dtype='string')))
    has_response=f.bytes_backward.gt(0)
    if 'packets_backward' in f:has_response |= f.packets_backward.gt(0)
    # REQ/INT can have response traffic. Literal no-reply/S0 with a response is
    # contradictory source evidence; do not silently erase that evidence.
    state_conflict=raw_state.isin(['no-reply','s0']) & has_response
    return pd.DataFrame({'invalid_required_value':invalid,
                         'bytes_packet_conflict':packet_conflict,
                         'state_counter_conflict':state_conflict},index=f.index)


TACTICS = ['Baseline Operations','TA0040: Impact','TA0043/TA0007: Recon & Discovery','TA0001/TA0004: Initial Access & PrivEsc','TA0006: Credential Access','TA0011/TA0008: C2 & Lateral Movement']


def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=lambda x:x.item() if hasattr(x,'item') else str(x)),encoding='utf8')


def read_tables(folder):
    # Source labels can be numeric (UNSW) or text (NSL/CIC). Preserve lexical
    # identifiers consistently across source tables, concatenation and CSV reload.
    types={name:'string' for name in ['record_id','label_raw','label_source_category','source_attack_category','orig_row_id','split_group','source_dataset','source_split','source_protocol','source_connection_state','source_service','service_basis','connection_state_basis','duration_unit','quality_issues','capture_time']}
    f,l,m=tuple(pd.read_csv(Path(folder)/(t+'.csv'),low_memory=False,dtype=types) for t in ['features','labels','metadata'])
    return f,l,normalize_boolean_metadata(m)


def validate_tables(f,l,m,allow_quarantined=False):
    require(len(f)>0 and len(f)==len(l)==len(m),'Empty tables or row-count mismatch')
    for name,frame in [('features',f),('labels',l),('metadata',m)]:
        require('record_id' in frame and frame.record_id.notna().all() and frame.record_id.astype(str).str.strip().ne('').all() and frame.record_id.is_unique,f'{name}: missing/duplicate record_id')
    require(f.record_id.equals(l.record_id) and f.record_id.equals(m.record_id),'Record IDs/order mismatch')
    require(not set(f.columns).intersection({'label','label_raw','label_source_category','attack_category','attack_category_id','label_binary','mitre_tactic','is_attack','source_dataset','sample_weight','data_split','is_synthetic'}),'Target/metadata leakage in features')
    require(set(CORE).issubset(f.columns),'Missing core columns')
    computed=quality_flags(f,m)
    eligible=~computed.any(axis=1)
    if 'quality_ok' in m:
        require(m.quality_ok.notna().all() and m.quality_ok.eq(eligible).all(),'quality_ok disagrees with recomputed quality rules')
        for col in computed:require(col in m and m[col].eq(computed[col]).all(),f'{col}: quality flag mismatch')
    require(allow_quarantined or eligible.all(),'Invalid duration/bytes/protocol or cross-field conflicts: quarantined records are not allowed in model splits')
    checked=f.loc[eligible] if allow_quarantined else f
    for c in NUMERIC:
        z=pd.to_numeric(checked[c],errors='coerce')
        require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: missing, infinite or negative input')
    for c,values in CATEGORIES.items():require(checked[c].notna().all() and checked[c].isin(values).all(),f'{c}: missing/unknown categories')
    response=checked.bytes_backward.gt(0)
    if 'packets_backward' in checked:response |= checked.packets_backward.gt(0)
    require(not (checked.conn_state_group.eq('no-reply') & response).any(),'no-reply contradicts backward traffic')
    for c in ['packets_forward','packets_backward','mean_packet_size','ttl_forward','ttl_backward','tcp_window_forward','tcp_window_backward']:
        if c in checked:
            z=checked[c].dropna()
            require(np.isfinite(z).all() and z.ge(0).all(),f'{c}: invalid measured extension')
    require('label_binary' in l and l.label_binary.isin([0,1]).all(),'Invalid binary labels')
    if 'attack_category_id' in l:
        require(l.attack_category_id.isin(range(6)).all(),'Invalid central classes')
        require(l.label_binary.eq(l.attack_category_id.ne(0).astype(int)).all(),'Binary/multiclass labels disagree')
        require(l.attack_category.eq(l.attack_category_id.map(dict(enumerate(CLASS_NAMES)))).all(),'Class name/ID mismatch')
    if 'sample_weight' in m:require(np.isfinite(m.sample_weight).all() and m.sample_weight.gt(0).all(),'Invalid sample weights')
    for flag in ['invalid_duration','invalid_bytes_forward','invalid_bytes_backward']:
        if flag in m:require(not m.loc[eligible,flag].astype(str).str.lower().isin(['true','1']).any(),f'{flag}: flagged invalid rows')


## 5. โหลดชุดฝึกและเตรียมอินพุต SMOTENC

ตรวจ split และคลาสเป้าหมาย โหลด preprocessor แล้วสร้างอินพุตผสมระหว่างคอลัมน์ตัวเลขและหมวดหมู่


In [ ]:
central_dir = PIPELINE_DIR / "CENTRAL"
split = SMOTE_SPLIT
target_class = TARGET_CLASS
target_samples = TARGET_SAMPLES
random_state = RANDOM_STATE

from imblearn.over_sampling import SMOTENC
require(split in ['train','train_full'],'Resampling only supports training splits')
o=Path(central_dir)/'output';f,l,m=read_tables(o/'splits'/split);validate_tables(f,l,m)
pre=joblib.load(o/'feature_engineering/preprocessor.joblib')
n=len(f);count=int(l.attack_category_id.eq(target_class).sum())
require(count>0,'Cannot synthesize an absent class')
# Distance uses three standardized log inputs and categorical attributes;
# the redundant ratio is recomputed after inverse transformation.
train=read_tables(o/'splits/train')[0]
numeric_scaler=StandardScaler().fit(np.log1p(train[NUMERIC].to_numpy(float)))
x=pd.DataFrame(numeric_scaler.transform(np.log1p(f[NUMERIC].to_numpy(float))),columns=NUMERIC)
for col in CATEGORIES:x[col]=f[col].to_numpy()


## 6. สร้างและตรวจตัวอย่างสังเคราะห์

เพิ่มตัวอย่างเมื่อคลาสเป้าหมายยังไม่ถึงจำนวนที่กำหนด คัดแถวที่ขัดกับกฎ connection state และเติมส่วนที่ขาดจากแถวจริงของคลาสนั้น


In [ ]:
rejected=0;fallback_rows=0
if count<target_samples:
    require(count>=2,'SMOTENC needs at least two minority samples')
    sampler=SMOTENC(categorical_features=[3,4,5],sampling_strategy={target_class:target_samples},k_neighbors=min(5,count-1),random_state=random_state)
    xr,yr=sampler.fit_resample(x,l.attack_category_id)
    z=xr.iloc[n:];k=len(z)
    synth=pd.DataFrame(np.maximum(np.expm1(numeric_scaler.inverse_transform(z[NUMERIC].to_numpy(float))),0),columns=NUMERIC)
    synth[['bytes_forward','bytes_backward']]=synth[['bytes_forward','bytes_backward']].round()
    for col in CATEGORIES:synth[col]=z[col].to_numpy()
    invalid_state=synth.conn_state_group.eq('no-reply') & synth.bytes_backward.gt(0)
    rejected=int(invalid_state.sum())
    synth=synth.loc[~invalid_state].reset_index(drop=True)
    # Mixed-category interpolation may violate state/counter constraints.
    # Reject those candidates; fill the exact deficit by transparent random
    # oversampling of valid minority inputs, never fabricate measurements.
    fallback_rows=k-len(synth)
    if fallback_rows:
        minority=f.loc[l.attack_category_id.eq(target_class),CORE]
        extra=minority.sample(n=fallback_rows,replace=True,random_state=random_state)
        synth=pd.concat([synth,extra],ignore_index=True)
    ids=[f'SMOTENC:{split}:C{target_class}:{random_state}:{i:08d}' for i in range(k)]
    synth.insert(0,'record_id',ids);synth=synth.reindex(columns=f.columns)
    sl=pd.DataFrame({'record_id':ids,'label_raw':'Synthetic_SMOTENC','label_source_category':CLASS_NAMES[target_class],'attack_category':CLASS_NAMES[target_class],'attack_category_id':target_class,'label_binary':int(target_class!=0),'mitre_tactic':TACTICS[target_class]})
    sm=pd.DataFrame(index=range(k),columns=m.columns)
    for col in m:
        if col.startswith('has_') or col.startswith('invalid_'):sm[col]=False
    for col in ['has_duration','has_protocol','has_directional_bytes','has_service','has_connection_state']:sm[col]=True
    sm['record_id']=ids;sm['source_dataset']='Synthetic_SMOTENC';sm['orig_row_id']=ids;sm['source_split']=split;sm['data_split']=split+'_smote';sm['split_group']=ids;sm['dedup_flag']=0;sm['is_kddtest21_view']=False
    sm['state_reclassified']=False;sm['service_identified']=False
    sm['service_basis']='synthetic_category';sm['connection_state_basis']='synthetic_category';sm['duration_unit']='seconds'
    # Treat all resampled rows equally by default; no 100,000x weight jump.
    f=pd.concat([f,synth],ignore_index=True);l=pd.concat([l,sl],ignore_index=True);m=pd.concat([m,sm],ignore_index=True)
else:k=0


## 7. ตรวจและบันทึกผล

ตรวจตารางสุดท้าย เขียน features, labels, metadata และฟีเจอร์ 19 คอลัมน์ พร้อม smote_report.json ในโฟลเดอร์ของ split ที่เลือก


In [ ]:
m['sample_weight']=1.0
m['is_synthetic']=np.arange(len(f))>=n;l['is_synthetic']=m.is_synthetic
if 'quality_ok' in m:
    flags=quality_flags(f,m)
    for col in flags:m[col]=flags[col]
    m['quality_ok']=~flags.any(axis=1)
    m.loc[m.is_synthetic,'quality_issues']=''
validate_tables(f,l,m)
dest=o/'splits'/(split+'_smote');dest.mkdir(parents=True,exist_ok=True)
for name,frame in [('features',f),('labels',l),('metadata',m)]:frame.to_csv(dest/(name+'.csv'),index=False)
transform_features(f,pre).to_csv(dest/'features_encoded_19.csv',index=False)
report={'method':'SMOTENC','source_split':split,'target_class_id':target_class,'target_samples':target_samples,'original_rows':n,'synthetic_rows':k,'total_rows':len(f),'original_target_count':count,'resampled_weight_policy':'uniform mean-one','preprocessor_fit_split':'train','preprocessor_fit_record_ids_sha256':pre['fit_record_ids_sha256'],'random_state':random_state,'rejected_candidates':rejected,'fallback_rows':fallback_rows,'fallback_method':'random_oversampling_of_valid_minority_rows'}
write_json(dest/'smote_report.json',report)
print(report)
